# 1장. 개발 환경 만들기

In [1]:
# [준비 셀] 모든 노트북의 첫 코드 셀입니다. 항상 먼저 실행하세요.
from pathlib import Path
import sys

ROOT = Path.cwd().resolve().parents[1]          # docs/manual → 프로젝트 루트
assert (ROOT / "src/bearing_kd").exists(), "docs/manual 폴더에서 노트북을 열어야 합니다"
CODE = ROOT / "docs/manual/code"                 # 챕터별 완성 스크립트 위치
sys.path.insert(0, str(CODE))                    # 스크립트의 함수를 import 해서 쓰기 위해
print("프로젝트 루트 확인 완료:", ROOT.name)

프로젝트 루트 확인 완료: vib-onnx


이 장에서는 코드를 실행할 **작업 공간**을 만듭니다. 한 번만 하면 됩니다.

## 1.1 가상환경이 뭔가요?

파이썬 프로그램은 `numpy`, `torch` 같은 **패키지(라이브러리)**를 가져다 씁니다. 그런데 프로젝트마다 필요한 버전이 다릅니다. 컴퓨터 전체에 한 버전만 깔면 다른 프로젝트가 깨질 수 있습니다.

**가상환경**은 프로젝트 전용 파이썬 + 패키지 상자입니다. 이 프로젝트의 가상환경은 `.venv-uv` 폴더이고, `source .venv-uv/bin/activate`로 "상자 안으로 들어가면" 그때부터 `python`은 이 상자 안의 파이썬을 뜻합니다. 터미널 프롬프트 앞에 `(.venv-uv)`가 붙으면 들어간 것입니다.

> 새 터미널 창을 열 때마다 `activate`를 다시 해야 합니다. 안 하면 `No module named ...` 에러가 납니다(부록 A).

## 1.2 설치 순서 (터미널에서)

이 프로젝트는 **Python 3.11**과 패키지 설치 도구 **uv**를 씁니다.

```bash
# 1) uv 설치 (macOS, Homebrew 사용 시)
brew install uv

# 2) 프로젝트 루트로 이동
cd vib-onnx

# 3) Python 3.11 가상환경 만들기
uv venv --python 3.11 .venv-uv
source .venv-uv/bin/activate

# 4) 패키지 설치 (검증된 버전 고정)
uv pip install -r requirements-lock.txt --override constraints-tested.txt

# 5) 이 매뉴얼용 Jupyter
uv pip install jupyterlab ipykernel
```

## 1.3 왜 `--override`가 필요한가요?

MOMENT(7장)의 패키지 `momentfm 0.1.4`는 오래된 `numpy`, `transformers`, `huggingface-hub` 버전을 **정확히 그 버전으로** 요구합니다. 그런데 LoRA(8장)에 쓰는 최신 `peft`는 더 새 버전을 요구합니다. 둘이 충돌해서 일반 `pip install`은 실패합니다.

`uv`의 override는 "MOMENT가 뭐라고 하든 이 버전을 써라"라고 지정하는 기능입니다. 이 프로젝트는 그렇게 설치한 조합(`constraints-tested.txt`)으로 전체 파이프라인을 검증했습니다. 그래서 나중에 `pip check`가 충돌 3개를 경고해도 정상입니다.

## 1.4 파이썬 버전과 가상환경 확인

In [2]:
import ch01_check_env
ch01_check_env.check_python()

Python 3.11.16 (Darwin arm64)
  가상환경 사용 중: 예


## 1.5 패키지 버전 확인

`[OK]`가 아닌 줄이 있으면 1.2의 4)번 설치를 다시 하세요.

In [3]:
ch01_check_env.check_packages()


패키지 버전 (설치됨 / 검증됨)
  [OK] torch            2.14.0     / 2.14.0
  [OK] momentfm         0.1.4      / 0.1.4
  [OK] peft             0.21.0     / 0.21.0
  [OK] numpy            2.4.6      / 2.4.6
  [OK] scipy            1.17.1     / 1.17.1
  [OK] transformers     4.57.6     / 4.57.6
  [OK] huggingface-hub  0.36.2     / 0.36.2
  [OK] onnx             1.23.0     / 1.23.0
  [OK] onnxruntime      1.30.0     / 1.30.0
  [OK] onnxscript       0.7.2      / 0.7.2
  [OK] matplotlib       3.11.2     / 3.11.2


## 1.6 연산 장치: CPU, CUDA, MPS

딥러닝 계산은 행렬 곱셈이 대부분이라 **GPU**가 있으면 훨씬 빠릅니다. PyTorch는 장치 이름으로 어디서 계산할지 정합니다.

| 장치 이름 | 하드웨어 | 확인 방법 |
|---|---|---|
| `cpu` | 모든 컴퓨터 | 항상 가능 |
| `cuda` | NVIDIA GPU (Windows/Linux, Colab) | `torch.cuda.is_available()` |
| `mps` | Apple Silicon Mac (M1–M4)의 GPU | `torch.backends.mps.is_available()` |

이 프로젝트에서 측정한 Teacher 학습 속도(16개 묶음 1번): **CPU 6.4초, MPS 3.1초**. Mac이라면 `mps`를 쓰세요.

MPS가 지원하지 않는 연산을 만나면 에러가 납니다. 환경변수 `PYTORCH_ENABLE_MPS_FALLBACK=1`을 설정하면 그 연산만 CPU로 넘겨 계산합니다.

In [4]:
ch01_check_env.check_device()


연산 장치
  CPU  : 항상 사용 가능 (스레드 4개)
  CUDA : 없음 (NVIDIA GPU 전용)
  MPS  : 사용 가능 (Apple Silicon GPU)
  → 추천 장치: mps  (시험 계산: 6 = 6 이면 정상)


텐서(tensor)는 "PyTorch의 다차원 배열"입니다. 장치를 지정해 만들면 그 장치의 메모리에 올라갑니다.

In [5]:
import torch

device = "mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu"
a = torch.randn(1000, 1000, device=device)   # 1000×1000 무작위 행렬
b = a @ a                                      # 행렬 곱셈 (GPU에서 계산)
print(device, b.shape, b.dtype)

mps torch.Size([1000, 1000]) torch.float32


## 1.7 라즈베리파이는 따로 준비합니다

라즈베리파이에서는 **학습을 하지 않고 ONNX 추론만** 합니다. 그래서 PyTorch나 MOMENT 없이 `numpy`와 `onnxruntime`만 있으면 됩니다. 이 프로젝트에서 검증한 라즈베리파이 5 환경은 Python 3.13입니다. 자세한 내용은 15장에서 다룹니다.

## 1장 완성 코드: `code/ch01_check_env.py`
```bash
python docs/manual/code/ch01_check_env.py
```

In [6]:
from IPython.display import Code
Code(filename=str(CODE / "ch01_check_env.py"), language="python")

"""1장 완성 코드: 개발 환경(파이썬, 패키지 버전, 연산 장치)을 점검한다.

실행 (프로젝트 루트에서, 가상환경 활성화 후):
    python docs/manual/code/ch01_check_env.py
"""

import importlib.metadata
import platform
import sys

# 이 프로젝트에서 검증한 버전 (constraints-tested.txt)
TESTED = {
    "torch": "2.14.0",
    "momentfm": "0.1.4",
    "peft": "0.21.0",
    "numpy": "2.4.6",
    "scipy": "1.17.1",
    "transformers": "4.57.6",
    "huggingface-hub": "0.36.2",
    "onnx": "1.23.0",
    "onnxruntime": "1.30.0",
    "onnxscript": "0.7.2",
    "matplotlib": "3.11.2",
}


def check_python():
    print(f"Python {platform.python_version()} ({platform.system()} {platform.machine()})")
    if sys.version_info[:2] != (3, 11):
        print("  ⚠ 학습 환경은 Python 3.11 기준입니다. (추론만 하는 라즈베리파이는 3.13도 가능)")
    in_venv = sys.prefix != sys.base_prefix
    print(f"  가상환경 사용 중: {'예' if in_venv else '아니오 ← 1장의 가상환경 활성화를 먼저 하세요'}")


def check_packages():
    print("\n패키지 버전 (설치됨 / 검증됨)")
    for name, tested in TESTED.items():
        try:
            installed = importlib.metadata.version(name)
        except importlib.metadata.PackageNotFoundError:
            installed = "없음"
        mark = "OK" if installed == tested else ("없음" if installed == "없음" else "다름")
        print(f"  [{mark:2s}] {name:16s} {installed:10s} / {tested}")


def check_device():
    import torch

    print("\n연산 장치")
    print(f"  CPU  : 항상 사용 가능 (스레드 {torch.get_num_threads()}개)")
    print(f"  CUDA : {'사용 가능 - ' + torch.cuda.get_device_name(0) if torch.cuda.is_available() else '없음 (NVIDIA GPU 전용)'}")
    print(f"  MPS  : {'사용 가능 (Apple Silicon GPU)' if torch.backends.mps.is_available() else '없음 (Apple Silicon Mac 전용)'}")

    device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
    x = torch.ones(2, 3, device=device)
    print(f"  → 추천 장치: {device}  (시험 계산: {x.sum().item():.0f} = 6 이면 정상)")


if __name__ == "__main__":
    check_python()
    check_packages()
    check_device()